# Corrected interior-expert mesh comparison

This notebook renders the corrected 2×2 experiment directly from its PLY meshes. Each mesh is cut open with the same plane and shown with the same camera, so the visible differences come from the SLAT model and decoder rather than the visualization.

| | Objective-1 SLAT | Interior-expert SLAT |
|---|---|---|
| **Objective-1 decoder** | top left | top right |
| **Adapted decoder** | bottom left | bottom right |


In [ ]:
from pathlib import Path

import numpy as np
import pyvista as pv
from IPython.display import display
from PIL import Image

RESULTS_DIR = Path("results/geometry_expert/corrected_interior_expert_v1/eval_test")
SEED = 42

# The +X half is removed. Use [0, 1, 0] or [0, 0, 1] for another axis.
CUT_NORMAL = np.array([1.0, 0.0, 0.0])
CUT_OFFSET = 0.0  # move the plane along CUT_NORMAL in model coordinates

METHODS = [
    ("O1 SLAT + O1 decoder", "objective1_slat_objective1_decoder"),
    ("Expert SLAT + O1 decoder", "expert_slat_objective1_decoder"),
    ("O1 SLAT + adapted decoder", "objective1_slat_adapted_decoder"),
    ("Expert SLAT + adapted decoder", "expert_slat_adapted_decoder"),
]

mesh_dirs = {
    label: RESULTS_DIR / "predictions" / method / f"seed_{SEED}" / "mesh"
    for label, method in METHODS
}
sample_ids = [
    line.strip()
    for line in (RESULTS_DIR / "selected_ids.txt").read_text().splitlines()
    if line.strip()
]

for sample_id in sample_ids:
    for label, mesh_dir in mesh_dirs.items():
        path = mesh_dir / f"{sample_id}.ply"
        if not path.is_file():
            raise FileNotFoundError(f"Missing {label} mesh: {path}")

print(f"Found all four predictions for {len(sample_ids)} test objects")

In [ ]:
def camera_for_cut(normal, center, object_size):
    up = np.array([0.0, 0.0, 1.0])
    if abs(normal @ up) > 0.9:
        up = np.array([0.0, 1.0, 0.0])
    side = np.cross(up, normal)
    position = center + object_size * (2.0 * normal + 0.65 * side + 0.45 * up)
    return [position.tolist(), center.tolist(), up.tolist()]


def render_comparison(sample_id):
    normal = CUT_NORMAL / np.linalg.norm(CUT_NORMAL)
    meshes = {
        label: pv.read(mesh_dir / f"{sample_id}.ply")
        for label, mesh_dir in mesh_dirs.items()
    }

    # Define the cut and camera once from the baseline, then reuse them everywhere.
    baseline = meshes["O1 SLAT + O1 decoder"]
    center = np.asarray(baseline.center)
    cut_origin = center + CUT_OFFSET * normal
    object_size = baseline.length

    plotter = pv.Plotter(shape=(2, 2), off_screen=True, window_size=(1600, 1200))
    for index, (label, _) in enumerate(METHODS):
        row, column = divmod(index, 2)
        cut_mesh = meshes[label].clip(normal=normal, origin=cut_origin, invert=True)

        plotter.subplot(row, column)
        plotter.set_background("white")
        plotter.add_mesh(
            cut_mesh,
            color="lightsteelblue",
            smooth_shading=True,
            ambient=0.25,
            diffuse=0.8,
            specular=0.15,
        )
        plotter.add_text(label, position="upper_left", color="black", font_size=12)

    plotter.link_views()
    plotter.camera_position = camera_for_cut(normal, center, object_size)
    plotter.camera.parallel_projection = True
    plotter.camera.parallel_scale = 0.42 * object_size
    image = plotter.screenshot(return_img=True)
    plotter.close()
    return Image.fromarray(image)


In [ ]:
for sample_id in sample_ids:
    print(sample_id)
    display(render_comparison(sample_id))